# Stock Market Data Analyzer — Educational Walkthrough

Welcome to the **Stock Market Data Analyzer** tutorial notebook. This notebook covers:
1. **Data Ingestion**: Loading historical prices from public APIs (Yahoo Finance), CSV files, and offline synthetic demo data.
2. **Data Cleaning & Normalization**: Parsing dates, converting numeric types, handling missing values, and chronological sorting.
3. **Return & Volatility Analysis**: Calculating daily percentage changes, cumulative returns, and annualized volatility.
4. **Technical Indicators**: Computing Simple Moving Averages (SMA 20/50), Exponential Moving Averages (EMA 12/26), RSI-14, MACD, and Bollinger Bands.
5. **Strategy Backtesting**: Simulating a Moving Average Crossover strategy with 1-bar execution delay and transaction friction.
6. **Visual Reporting**: Rendering price trends, moving average envelopes, and return distributions.

> **Disclaimer:** All data and indicators in this notebook are for educational purposes only and do NOT constitute financial advice.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure project root src is importable
repo_root = Path.cwd().parent
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

from stock_analyzer.data import demo_history, clean_prices, load_prices
from stock_analyzer.analysis import analyze_prices, build_summary, run_backtest

sns.set_theme(style="whitegrid")
print("Libraries and stock_analyzer modules imported successfully!")

## 1. Data Ingestion & Deterministic Offline Demo
We can generate a reproducible, offline synthetic dataset for `AAPL` using our SHA-256 Geometric Brownian Motion generator.

In [ ]:
# Load 252 business days of deterministic synthetic data
df_raw = demo_history("AAPL", days=252)
print(f"Loaded {len(df_raw)} bars.")
df_raw.head()

## 2. Technical Indicator Computations
Next, we calculate the technical indicators: SMA 20, SMA 50, EMA 12, EMA 26, MACD, Wilder's RSI 14, and Bollinger Bands.

In [ ]:
df_analyzed = analyze_prices(df_raw, fast_window=20, slow_window=50)
print("Indicators calculated:")
df_analyzed[["date", "close", "daily_return_pct", "sma_20", "sma_50", "rsi_14", "macd", "bb_upper", "bb_lower"]].tail()

## 3. Visualization: Price Trend & Moving Averages

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(df_analyzed["date"], df_analyzed["close"], label="Close Price", color="#64748b", alpha=0.7)
plt.plot(df_analyzed["date"], df_analyzed["sma_20"], label="SMA 20", color="#10b981", linewidth=2)
plt.plot(df_analyzed["date"], df_analyzed["sma_50"], label="SMA 50", color="#8b5cf6", linewidth=2)
plt.fill_between(df_analyzed["date"], df_analyzed["bb_lower"], df_analyzed["bb_upper"], color="#38bdf8", alpha=0.15, label="Bollinger Bands (2σ)")
plt.title("AAPL - Closing Price & Indicator Envelopes", fontsize=14, fontweight="bold")
plt.xlabel("Date")
plt.ylabel("Price ($)")
plt.legend()
plt.show()

## 4. Moving Average Crossover Backtesting Simulation
We simulate a long-or-cash strategy where positions are entered when Fast SMA > Slow SMA.
- **Execution Shift**: Position is shifted by 1 bar (`shift(1)`) to avoid lookahead bias.
- **Transaction Friction**: 10 basis points (0.10%) per trade turnover.

In [ ]:
bt = run_backtest(df_analyzed, fast_window=20, slow_window=50, initial_cash=10000.0, transaction_cost_bps=10.0)
metrics = bt["metrics"]
print(f"Initial Cash        : ${metrics['initial_cash']:,.2f}")
print(f"Ending Strategy Cash: ${metrics['ending_value']:,.2f} ({metrics['total_return_pct']:+.2f}%)")
print(f"Benchmark Buy & Hold: {metrics['buy_and_hold_pct']:+.2f}%")
print(f"CAGR                : {metrics['cagr_pct']:+.2f}%")
print(f"Max Drawdown        : {metrics['max_drawdown_pct']:.2f}%")
print(f"Sharpe Ratio        : {metrics['sharpe_ratio']:.2f}")
print(f"Trades Executed     : {metrics['trade_count']}")

In [ ]:
df_eq = pd.DataFrame(bt["equity_curve"])
df_eq["date"] = pd.to_datetime(df_eq["date"])

plt.figure(figsize=(12, 5))
plt.plot(df_eq["date"], df_eq["strategy"], label="SMA Crossover Strategy", color="#2563eb", linewidth=2)
plt.plot(df_eq["date"], df_eq["benchmark"], label="Buy & Hold Benchmark", color="#94a3b8", linestyle="--")
plt.title("AAPL - Strategy Equity Curve vs. Benchmark", fontsize=14, fontweight="bold")
plt.xlabel("Date")
plt.ylabel("Portfolio Value ($)")
plt.legend()
plt.show()